#  **Deduplication**

In [0]:
from pyspark.sql import functions as F

# =========================================================
# PATHS
# =========================================================

bronze_path = "/Volumes/aeroflux/bronze_layer/bronze_data"

deduplicated_path = (
    "/Volumes/aeroflux/silver_layer/"
    "deduplication-layer/"
)

checkpoint_path = (
    "/Volumes/aeroflux/silver_layer/"
    "checkpoints/deduplication/"
)


# =========================================================
# 1. READ BRONZE PARQUET AS STREAM
# =========================================================

df = (
    spark.readStream
        .format("delta")
        .load(bronze_path)
)


# =========================================================
# 2. NORMALIZE AIRCRAFT ID + EVENT TIME
# =========================================================

df = (
    df
    .withColumn(
        "icao24",
        F.lower(F.trim(F.col("icao24")))
    )
    .withColumn(
        "event_time",
        F.to_timestamp(F.col("time_position"))
    )
    .filter(
        F.col("icao24").isNotNull() &
        F.col("event_time").isNotNull()
    )
)


# =========================================================
# 3. CREATE DETERMINISTIC OBSERVATION ID
# =========================================================

dedup_columns = [
    "icao24",
    "event_time",
    "latitude",
    "longitude",
    "baro_altitude_m",
    "velocity_mps",
    "true_track",
    "vertical_rate_mps"
]

df = df.withColumn(
    "observation_id",
    F.sha2(
        F.concat_ws(
            "||",
            *[
                F.coalesce(
                    F.col(column).cast("string"),
                    F.lit("<NULL>")
                )
                for column in dedup_columns
            ]
        ),
        256
    )
)


# =========================================================
# 4. WATERMARK + STREAMING DEDUPLICATION
# =========================================================

df_silver = (
    df
    .withWatermark(
        "event_time",
        "10 minutes"
    )
    .dropDuplicates(
        ["observation_id"]
    )
)


# =========================================================
# 5. WRITE DEDUPLICATED DATA TO SILVER
# =========================================================

query = (
    df_silver
    .writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        checkpoint_path
    )
    .trigger(availableNow=True)
    .start(deduplicated_path)
)


# =========================================================
# 6. KEEP STREAM RUNNING
# =========================================================

query.awaitTermination()

# **Great Expectations Quality Gate and Filtering Bad Records**

In [0]:
from pyspark.sql import functions as F

# =========================================================
# PATHS
# =========================================================

deduplicated_path = (
    "/Volumes/aeroflux/silver_layer/deduplication-layer"
)

clean_path = (
    "/Volumes/aeroflux/silver_layer/quality-gate"
)

bad_records_path = (
    "/Volumes/aeroflux/silver_layer/bad-records"
)

checkpoint_clean = (
    "/Volumes/aeroflux/silver_layer/checkpoints/quality-gate"
)

checkpoint_bad = (
    "/Volumes/aeroflux/silver_layer/checkpoints/bad-records"
)


# =========================================================
# 1. READ DEDUPLICATED DELTA
# =========================================================

df = (
    spark.readStream
        .format("delta")
        .load(deduplicated_path)
)


# =========================================================
# 2. NORMALIZE BASIC FIELDS
# =========================================================

df = (
    df
    .withColumn(
        "icao24",
        F.lower(F.trim(F.col("icao24")))
    )
)


# =========================================================
# 3. CREATE DATA-QUALITY FLAGS
# =========================================================

df = (
    df

    # -----------------------------------------------------
    # ICAO24
    # 6 hexadecimal characters
    # -----------------------------------------------------

    .withColumn(
        "dq_invalid_icao24",
        ~F.col("icao24").rlike("^[0-9a-f]{6}$")
    )

    # -----------------------------------------------------
    # EVENT TIME
    # -----------------------------------------------------

    .withColumn(
        "dq_invalid_event_time",
        F.col("event_time").isNull()
    )

    # -----------------------------------------------------
    # LATITUDE
    # -----------------------------------------------------

    .withColumn(
        "dq_invalid_latitude",
        (
            F.col("latitude").isNotNull()
            & (
                (F.col("latitude") < -90)
                | (F.col("latitude") > 90)
            )
        )
    )

    # -----------------------------------------------------
    # LONGITUDE
    # -----------------------------------------------------

    .withColumn(
        "dq_invalid_longitude",
        (
            F.col("longitude").isNotNull()
            & (
                (F.col("longitude") < -180)
                | (F.col("longitude") > 180)
            )
        )
    )

    # -----------------------------------------------------
    # ALTITUDE
    # -----------------------------------------------------

    .withColumn(
        "dq_invalid_altitude",
        (
            F.col("baro_altitude_m").isNotNull()
            & (
                (F.col("baro_altitude_m") < -1000)
                | (F.col("baro_altitude_m") > 25000)
            )
        )
    )

    # -----------------------------------------------------
    # VELOCITY
    # -----------------------------------------------------

    .withColumn(
        "dq_invalid_velocity",
        (
            F.col("velocity_mps").isNotNull()
            & (F.col("velocity_mps") < 0)
        )
    )

    # -----------------------------------------------------
    # TRUE TRACK
    # -----------------------------------------------------

    .withColumn(
        "dq_invalid_track",
        (
            F.col("true_track").isNotNull()
            & (
                (F.col("true_track") < 0)
                | (F.col("true_track") > 360)
            )
        )
    )

    # -----------------------------------------------------
    # VERTICAL RATE
    # -----------------------------------------------------

    .withColumn(
        "dq_invalid_vertical_rate",
        (
            F.col("vertical_rate_mps").isNotNull()
            & (
                (F.col("vertical_rate_mps") < -100)
                | (F.col("vertical_rate_mps") > 100)
            )
        )
    )
)


# =========================================================
# 4. NULLIFY INVALID NON-CRITICAL VALUES
# =========================================================

df_cleaned = (
    df

    .withColumn(
        "latitude",
        F.when(
            F.col("dq_invalid_latitude"),
            F.lit(None).cast("double")
        ).otherwise(F.col("latitude"))
    )

    .withColumn(
        "longitude",
        F.when(
            F.col("dq_invalid_longitude"),
            F.lit(None).cast("double")
        ).otherwise(F.col("longitude"))
    )

    .withColumn(
        "baro_altitude_m",
        F.when(
            F.col("dq_invalid_altitude"),
            F.lit(None).cast("double")
        ).otherwise(F.col("baro_altitude_m"))
    )

    .withColumn(
        "velocity_mps",
        F.when(
            F.col("dq_invalid_velocity"),
            F.lit(None).cast("double")
        ).otherwise(F.col("velocity_mps"))
    )

    .withColumn(
        "true_track",
        F.when(
            F.col("dq_invalid_track"),
            F.lit(None).cast("double")
        ).otherwise(F.col("true_track"))
    )

    .withColumn(
        "vertical_rate_mps",
        F.when(
            F.col("dq_invalid_vertical_rate"),
            F.lit(None).cast("double")
        ).otherwise(F.col("vertical_rate_mps"))
    )
)


# =========================================================
# 5. DEFINE CRITICAL QUALITY FAILURE
# =========================================================

df_quality = df_cleaned.withColumn(
    "quality_status",
    F.when(
        F.col("dq_invalid_icao24")
        | F.col("dq_invalid_event_time"),
        F.lit("BAD")
    ).otherwise(
        F.lit("VALID")
    )
)


# =========================================================
# 6. ADD QUALITY METADATA
# =========================================================

df_quality = (
    df_quality
    .withColumn(
        "quality_checked_at",
        F.current_timestamp()
    )
    .withColumn(
        "quality_rule_version",
        F.lit("v1.0")
    )
)


# =========================================================
# 7. VALID RECORDS
# =========================================================

valid_records = (
    df_quality
    .filter(
        F.col("quality_status") == "VALID"
    )
    .drop(
        "dq_invalid_icao24",
        "dq_invalid_event_time",
        "dq_invalid_latitude",
        "dq_invalid_longitude",
        "dq_invalid_altitude",
        "dq_invalid_velocity",
        "dq_invalid_track",
        "dq_invalid_vertical_rate"
    )
)


# =========================================================
# 8. BAD RECORDS
# =========================================================

bad_records = (
    df_quality
    .filter(
        F.col("quality_status") == "BAD"
    )
)


# =========================================================
# 9. WRITE VALID RECORDS
# =========================================================

clean_query = (
    valid_records
    .writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        checkpoint_clean
    )
    .trigger(
        availableNow=True
    )
    .start(clean_path)
)


# =========================================================
# 10. WRITE BAD RECORDS
# =========================================================

bad_query = (
    bad_records
    .writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        checkpoint_bad
    )
    .trigger(
        availableNow=True
    )
    .start(bad_records_path)
)


# =========================================================
# 11. WAIT FOR BOTH STREAMS
# =========================================================

clean_query.awaitTermination()
bad_query.awaitTermination()

# **H3 Spatial Indexing**

In [0]:
from pyspark.sql import functions as F
from pyspark.databricks.sql import functions as dbf


# =========================================================
# PATHS
# =========================================================

quality_gate_path = (
    "/Volumes/aeroflux/silver_layer/quality-gate/"
)

h3_path = (
    "/Volumes/aeroflux/silver_layer/spatial-indexed/"
)

checkpoint_path = (
    "/Volumes/aeroflux/silver_layer/checkpoints/spatial-indexed/"
)


# =========================================================
# H3 CONFIGURATION
# =========================================================

H3_RES_COARSE = 6
H3_RES_OPERATIONAL = 8
H3_RES_FINE = 10

# Local neighborhood around the operational cell
H3_K_RING = 1


# =========================================================
# 1. READ QUALITY-GATED SILVER DATA
# =========================================================

df = (
    spark.readStream
        .format("delta")
        .load(quality_gate_path)
)


# =========================================================
# 2. DETERMINE WHETHER SPATIAL DATA IS USABLE
# =========================================================

df = df.withColumn(
    "spatial_valid",
    F.col("latitude").isNotNull()
    & F.col("longitude").isNotNull()
)


# =========================================================
# 3. H3 RESOLUTION 6
#    COARSE / REGIONAL INDEX
# =========================================================

df = df.withColumn(
    "h3_res_6",
    F.when(
        F.col("spatial_valid"),
        dbf.h3_longlatash3(
            F.col("longitude"),
            F.col("latitude"),
            F.lit(H3_RES_COARSE)
        )
    )
)


# =========================================================
# 4. H3 RESOLUTION 8
#    OPERATIONAL / TRAFFIC-DENSITY INDEX
# =========================================================

df = df.withColumn(
    "h3_res_8",
    F.when(
        F.col("spatial_valid"),
        dbf.h3_longlatash3(
            F.col("longitude"),
            F.col("latitude"),
            F.lit(H3_RES_OPERATIONAL)
        )
    )
)


# =========================================================
# 5. H3 RESOLUTION 10
#    HIGH-RESOLUTION / LOCAL INDEX
# =========================================================

df = df.withColumn(
    "h3_res_10",
    F.when(
        F.col("spatial_valid"),
        dbf.h3_longlatash3(
            F.col("longitude"),
            F.col("latitude"),
            F.lit(H3_RES_FINE)
        )
    )
)


# =========================================================
# 6. USE H3-10 AS THE FINE-GRAINED MASTER CELL
# =========================================================

df = df.withColumn(
    "h3_cell",
    F.col("h3_res_10")
)


# =========================================================
# 7. H3 VALIDITY
# =========================================================

df = df.withColumn(
    "h3_valid",
    F.when(
        F.col("h3_cell").isNotNull(),
        dbf.h3_isvalid(F.col("h3_cell"))
    ).otherwise(F.lit(False))
)


# =========================================================
# 8. PENTAGON FLAG
# =========================================================

df = df.withColumn(
    "h3_is_pentagon",
    F.when(
        F.col("h3_cell").isNotNull(),
        dbf.h3_ispentagon(F.col("h3_cell"))
    ).otherwise(F.lit(False))
)


# =========================================================
# 9. VERIFY H3 RESOLUTION
# =========================================================

df = df.withColumn(
    "h3_resolution",
    F.when(
        F.col("h3_cell").isNotNull(),
        dbf.h3_resolution(F.col("h3_cell"))
    )
)


# =========================================================
# 10. DERIVE PARENTS FROM FINE INDEX
#
# h3_res_10
#      ↓
# h3_res_8
#      ↓
# h3_res_6
# =========================================================

df = (
    df
    .withColumn(
        "h3_parent_res_8",
        F.when(
            F.col("h3_cell").isNotNull(),
            dbf.h3_toparent(
                F.col("h3_cell"),
                F.lit(H3_RES_OPERATIONAL)
            )
        )
    )
    .withColumn(
        "h3_parent_res_6",
        F.when(
            F.col("h3_cell").isNotNull(),
            dbf.h3_toparent(
                F.col("h3_cell"),
                F.lit(H3_RES_COARSE)
            )
        )
    )
)


# =========================================================
# 11. H3 STRING REPRESENTATIONS
#
# Useful for visualization/debugging.
# Keep BIGINT versions for analytics.
# =========================================================

df = (
    df
    .withColumn(
        "h3_cell_string",
        F.when(
            F.col("h3_cell").isNotNull(),
            dbf.h3_h3tostring(F.col("h3_cell"))
        )
    )
    .withColumn(
        "h3_res_8_string",
        F.when(
            F.col("h3_res_8").isNotNull(),
            dbf.h3_h3tostring(F.col("h3_res_8"))
        )
    )
)


# =========================================================
# 12. H3 CELL CENTER
#
# Returns WKT POINT:
# POINT(longitude latitude)
# =========================================================

df = df.withColumn(
    "h3_center_wkt",
    F.when(
        F.col("h3_cell").isNotNull(),
        dbf.h3_centeraswkt(F.col("h3_cell"))
    )
)


# =========================================================
# 13. LOCAL H3 NEIGHBORHOOD
#
# k = 1:
# current cell + neighboring cells
# =========================================================

df = df.withColumn(
    "h3_k_ring_1",
    F.when(
        F.col("h3_cell").isNotNull(),
        dbf.h3_kring(
            F.col("h3_cell"),
            F.lit(H3_K_RING)
        )
    )
)


# =========================================================
# 14. SPATIAL QUALITY STATUS
# =========================================================

df = df.withColumn(
    "spatial_quality_status",
    F.when(
        ~F.col("spatial_valid"),
        F.lit("NO_COORDINATES")
    )
    .when(
        ~F.col("h3_valid"),
        F.lit("INVALID_H3")
    )
    .when(
        F.col("h3_is_pentagon"),
        F.lit("PENTAGON")
    )
    .otherwise(
        F.lit("VALID")
    )
)


# =========================================================
# 15. SPATIAL INDEX VERSION
# =========================================================

df = df.withColumn(
    "spatial_index_version",
    F.lit("h3-r6-r8-r10-v1")
)


# =========================================================
# 16. INDEXING TIMESTAMP
# =========================================================

df = df.withColumn(
    "spatial_indexed_at",
    F.current_timestamp()
)


# =========================================================
# 17. WRITE H3-ENRICHED SILVER
# =========================================================

query = (
    df
    .writeStream
    .format("delta")
    .outputMode("append")
    .option(
        "checkpointLocation",
        checkpoint_path
    )
    .trigger(
        availableNow=True
    )
    .start(h3_path)
)


# =========================================================
# 18. WAIT
# =========================================================

query.awaitTermination()

# **Testing : Map Aircraft to JFK/LHR**

In [0]:
from pyspark.sql.functions import udf
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType
)

import math


# =========================================================
# AIRPORT REFERENCE DATA
# =========================================================

AIRPORTS = {
    "JFK": (40.6413, -73.7781),
    "LHR": (51.4700, -0.4543)
}


# =========================================================
# HAVERSINE + NEAREST AIRPORT UDF
# =========================================================

@udf(
    returnType=StructType([
        StructField("airport", StringType(), True),
        StructField("distance_km", DoubleType(), True)
    ])
)
def nearest_airport(latitude, longitude):

    if latitude is None or longitude is None:
        return (None, None)

    def haversine(lat1, lon1, lat2, lon2):

        R = 6371.0  # Earth radius in km

        lat1 = math.radians(lat1)
        lon1 = math.radians(lon1)
        lat2 = math.radians(lat2)
        lon2 = math.radians(lon2)

        dlat = lat2 - lat1
        dlon = lon2 - lon1

        a = (
            math.sin(dlat / 2) ** 2
            + math.cos(lat1)
            * math.cos(lat2)
            * math.sin(dlon / 2) ** 2
        )

        # Protect against floating-point drift
        a = min(1.0, max(0.0, a))

        return 2 * R * math.asin(math.sqrt(a))

    distances = {
        airport: haversine(
            latitude,
            longitude,
            coords[0],
            coords[1]
        )
        for airport, coords in AIRPORTS.items()
    }

    nearest = min(
        distances,
        key=distances.get
    )

    return (
        nearest,
        float(distances[nearest])
    )

from pyspark.sql import functions as F

test_df = (
    spark.read
    .format("delta")
    .load("/Volumes/aeroflux/silver_layer/spatial-indexed/")
)

result = (
    test_df
    .withColumn(
        "nearest_airport",
        nearest_airport(
            F.col("latitude"),
            F.col("longitude")
        )
    )
    .select(
        "icao24",
        "event_time",
        "latitude",
        "longitude",
        F.col("nearest_airport.airport").alias("nearest_airport"),
        F.col("nearest_airport.distance_km").alias("distance_to_airport_km")
    )
)

display(result.limit(20))

icao24,event_time,latitude,longitude,nearest_airport,distance_to_airport_km
4bc8c7,2026-09-22T08:00:54.000Z,49.266,21.1383,LHR,1545.130691127504
e80269,2026-09-22T08:00:54.000Z,-22.3565,-47.7766,JFK,7503.517910978813
3455d7,2026-09-22T08:01:09.000Z,43.3632,0.0883,LHR,902.3520362493226
51118a,2026-09-25T10:22:32.000Z,43.2734,22.0842,LHR,1915.4483348677834
0d1314,2026-09-27T06:18:59.000Z,31.8138,-115.0295,JFK,3790.9578169536353
801638,2026-09-25T09:51:34.000Z,17.9376,81.1057,LHR,7879.034662047975
39de51,2026-09-25T09:51:35.000Z,42.2874,-1.7088,LHR,1025.4588571560125
4bc8d9,2026-09-25T09:51:35.000Z,47.6512,19.272,LHR,1479.5786890578777
80162f,2026-09-25T09:40:59.000Z,23.0297,88.6046,LHR,7962.88607960875
4bc8d5,2026-09-25T10:21:06.000Z,47.6573,17.9338,LHR,1387.9427695850186


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# =========================================================
# AIRPORT REFERENCE DATA
# =========================================================

AIRPORTS = [
    ("JFK", 40.6413, -73.7781),
    ("LHR", 51.4700, -0.4543),
    ("CDG", 49.0097, 2.5479),
    ("DXB", 25.2532, 55.3657),
    ("DEL", 28.5562, 77.1000),
    ("BOM", 19.0896, 72.8656),
]

airport_df = spark.createDataFrame(
    AIRPORTS,
    [
        "airport",
        "airport_lat",
        "airport_lon"
    ]
)
flight_df = (
    spark.read
        .format("delta")
        .load(
            "/Volumes/aeroflux/silver_layer/"
            "spatial-indexed/"
        )
        .filter(
            F.col("latitude").isNotNull() &
            F.col("longitude").isNotNull()
        )
)
candidate_df = (
    flight_df
    .crossJoin(
        F.broadcast(airport_df)
    )
)
earth_radius_km = 6371.0

lat1 = F.radians(F.col("latitude"))
lon1 = F.radians(F.col("longitude"))

lat2 = F.radians(F.col("airport_lat"))
lon2 = F.radians(F.col("airport_lon"))

dlat = lat2 - lat1
dlon = lon2 - lon1

a = (
    F.pow(F.sin(dlat / 2), 2)
    +
    F.cos(lat1)
    * F.cos(lat2)
    * F.pow(F.sin(dlon / 2), 2)
)

# Floating-point safety
a_safe = F.least(
    F.lit(1.0),
    F.greatest(
        F.lit(0.0),
        a
    )
)

distance_km = (
    F.lit(2.0)
    * F.lit(earth_radius_km)
    * F.asin(
        F.sqrt(a_safe)
    )
)

candidate_df = candidate_df.withColumn(
    "distance_to_airport_km",
    distance_km
)
window_spec = (
    Window
    .partitionBy("observation_id")
    .orderBy(
        F.col("distance_to_airport_km").asc()
    )
)

nearest_airport_df = (
    candidate_df
    .withColumn(
        "airport_rank",
        F.row_number().over(window_spec)
    )
    .filter(
        F.col("airport_rank") == 1
    )
    .drop(
        "airport_rank",
        "airport_lat",
        "airport_lon"
    )
)
display(
    nearest_airport_df.select(
        "icao24",
        "event_time",
        "latitude",
        "longitude",
        "airport",
        "distance_to_airport_km",
        "h3_cell"
    ).limit(50)
)

icao24,event_time,latitude,longitude,airport,distance_to_airport_km,h3_cell
a5d203,2026-09-27T10:00:53.000Z,61.477,-23.5744,LHR,1787.5474716915749,621625094566379519
4acb01,2026-09-27T10:03:10.000Z,54.0736,3.476,LHR,391.95564213849616,621944169334636543
80161d,2026-09-25T09:22:44.000Z,13.9825,78.109,BOM,796.6242161252903,623212755992346623
4bc8c3,2026-09-27T09:49:08.000Z,41.5385,28.8298,CDG,2203.988992006256,622038366001332223
a4b234,2026-09-25T06:17:07.000Z,42.5751,-84.0619,JFK,881.0038440671027,622188162681503743
505f15,2026-09-27T10:08:15.000Z,42.588,26.0804,CDG,1949.1743028319872,622040623059599359
39de51,2026-09-25T05:58:04.000Z,42.3356,1.5486,CDG,746.1576902262735,622506248902082559
4bc8d5,2026-09-27T10:07:50.000Z,51.7185,22.7287,CDG,1457.797049765752,622027642296369151
3442cc,2026-09-25T11:54:06.000Z,35.4857,-3.0626,CDG,1571.989912064066,622485596219768831
39de54,2026-09-25T11:58:28.000Z,45.526,8.6223,CDG,599.7741744734179,622052598278914047


# **Trajectory enrichment**

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# =========================================================
# AEROFLUX - FINAL TRAJECTORY ENRICHMENT
# =========================================================
#
# INPUT
# -----
# /Volumes/aeroflux/silver_layer/spatial-indexed/
#
# OUTPUT
# ------
# /Volumes/aeroflux/silver_layer/trajectory-enriched/
#
# PURPOSE
# -------
# Build the canonical aircraft trajectory layer:
#
#   aircraft observations
#          ↓
#   previous state
#          ↓
#   temporal validity
#          ↓
#   kinematic features
#          ↓
#   H3 transitions
#          ↓
#   flight sessions
#          ↓
#   continuous trajectory segments
#          ↓
#   segment-level path efficiency
#
# IMPORTANT
# ---------
# This layer is intentionally a BATCH Delta transformation
# because it relies on lag() and ordered window operations.
#
# Current Free Edition architecture:
#
#   Bronze/Silver streaming
#          ↓
#   Delta
#          ↓
#   Batch trajectory enrichment
#
# Later Airflow can orchestrate this job.
#
# =========================================================


# =========================================================
# 1. PATHS
# =========================================================

SOURCE_PATH = (
    "/Volumes/aeroflux/silver_layer/spatial-indexed"
)

TRAJECTORY_PATH = (
    "/Volumes/aeroflux/silver_layer/trajectory-enrichment"
)


# =========================================================
# 2. CONFIGURATION
# =========================================================

EARTH_RADIUS_KM = 6371.0

# New flight/session after this gap.
SESSION_GAP_SECONDS = 30 * 60

# Instantaneous trajectory features are only calculated
# over reasonably close observations.
MIN_KINEMATIC_GAP_SECONDS = 5
MAX_KINEMATIC_GAP_SECONDS = 120

# Speed consistency is a diagnostic, not a rejection rule.
SPEED_ERROR_THRESHOLD = 0.30


# =========================================================
# 3. READ H3-ENRICHED SILVER
# =========================================================

df = (
    spark.read
    .format("delta")
    .load(SOURCE_PATH)
)


# =========================================================
# 4. SCHEMA VALIDATION
# =========================================================

required_columns = [
    "observation_id",
    "icao24",
    "event_time",
    "latitude",
    "longitude",
    "baro_altitude_m",
    "velocity_mps",
    "true_track",
    "vertical_rate_mps",
    "h3_res_8",
    "h3_res_8_string",
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        "Missing required columns: "
        + ", ".join(missing_columns)
    )


# =========================================================
# 5. OPTIONAL on_ground COLUMN
# =========================================================

if "on_ground" not in df.columns:
    df = df.withColumn(
        "on_ground",
        F.lit(None).cast("boolean")
    )


# =========================================================
# 6. NORMALIZE ICAO24
# =========================================================

df = df.withColumn(
    "icao24",
    F.lower(
        F.trim(
            F.col("icao24")
        )
    )
)


# =========================================================
# 7. AIRCRAFT CHRONOLOGICAL WINDOW
# =========================================================
#
# observation_id provides a deterministic tie-breaker
# when two observations have identical event_time.
# =========================================================

trajectory_window = (
    Window
    .partitionBy("icao24")
    .orderBy(
        F.col("event_time").asc(),
        F.col("observation_id").asc()
    )
)


# =========================================================
# 8. PREVIOUS AIRCRAFT STATE
# =========================================================

df = (
    df

    .withColumn(
        "previous_event_time",
        F.lag("event_time").over(
            trajectory_window
        )
    )

    .withColumn(
        "previous_latitude",
        F.lag("latitude").over(
            trajectory_window
        )
    )

    .withColumn(
        "previous_longitude",
        F.lag("longitude").over(
            trajectory_window
        )
    )

    .withColumn(
        "previous_altitude_m",
        F.lag("baro_altitude_m").over(
            trajectory_window
        )
    )

    .withColumn(
        "previous_velocity_mps",
        F.lag("velocity_mps").over(
            trajectory_window
        )
    )

    .withColumn(
        "previous_true_track",
        F.lag("true_track").over(
            trajectory_window
        )
    )

    .withColumn(
        "previous_vertical_rate_mps",
        F.lag("vertical_rate_mps").over(
            trajectory_window
        )
    )

    .withColumn(
        "previous_h3_res_8",
        F.lag("h3_res_8").over(
            trajectory_window
        )
    )

    .withColumn(
        "previous_h3_res_8_string",
        F.lag("h3_res_8_string").over(
            trajectory_window
        )
    )

    .withColumn(
        "previous_on_ground",
        F.lag("on_ground").over(
            trajectory_window
        )
    )
)


# =========================================================
# 9. TIME DELTA
# =========================================================

df = df.withColumn(
    "time_delta_seconds",
    F.when(
        F.col("previous_event_time").isNotNull(),
        (
            F.col("event_time").cast("long")
            -
            F.col("previous_event_time").cast("long")
        ).cast("double")
    )
)


# =========================================================
# 10. TRAJECTORY STEP STATUS
# =========================================================
#
# FIRST_OBSERVATION
# TOO_CLOSE
# VALID
# GAP_TOO_LARGE
# INVALID_TIME_ORDER
#
# TOO_CLOSE is NOT treated as a trajectory break.
# =========================================================

df = df.withColumn(
    "trajectory_step_status",
    F.when(
        F.col("previous_event_time").isNull(),
        F.lit("FIRST_OBSERVATION")
    )
    .when(
        F.col("time_delta_seconds") <= 0,
        F.lit("INVALID_TIME_ORDER")
    )
    .when(
        F.col("time_delta_seconds")
        < MIN_KINEMATIC_GAP_SECONDS,
        F.lit("TOO_CLOSE")
    )
    .when(
        F.col("time_delta_seconds")
        > MAX_KINEMATIC_GAP_SECONDS,
        F.lit("GAP_TOO_LARGE")
    )
    .otherwise(
        F.lit("VALID")
    )
)


# =========================================================
# 11. KINEMATIC STEP VALIDITY
# =========================================================

df = df.withColumn(
    "trajectory_step_valid",
    (
        (F.col("trajectory_step_status") == "VALID")
        &
        F.col("previous_latitude").isNotNull()
        &
        F.col("previous_longitude").isNotNull()
        &
        F.col("latitude").isNotNull()
        &
        F.col("longitude").isNotNull()
    )
)


# =========================================================
# 12. HAVERSINE STEP DISTANCE
# =========================================================

lat1 = F.radians(
    F.col("previous_latitude")
)

lon1 = F.radians(
    F.col("previous_longitude")
)

lat2 = F.radians(
    F.col("latitude")
)

lon2 = F.radians(
    F.col("longitude")
)

dlat = lat2 - lat1
dlon = lon2 - lon1

a = (
    F.pow(
        F.sin(dlat / 2),
        2
    )
    +
    F.cos(lat1)
    * F.cos(lat2)
    * F.pow(
        F.sin(dlon / 2),
        2
    )
)

a_safe = F.least(
    F.lit(1.0),
    F.greatest(
        F.lit(0.0),
        a
    )
)

step_distance_km_expr = (
    F.lit(2.0)
    * F.lit(EARTH_RADIUS_KM)
    * F.asin(
        F.sqrt(a_safe)
    )
)


df = df.withColumn(
    "step_distance_km",
    F.when(
        F.col("trajectory_step_valid"),
        step_distance_km_expr
    )
)


# =========================================================
# 13. DERIVED GROUND SPEED
# =========================================================

df = df.withColumn(
    "derived_speed_mps",
    F.when(
        F.col("trajectory_step_valid")
        &
        (F.col("time_delta_seconds") > 0),
        (
            F.col("step_distance_km")
            * 1000.0
            /
            F.col("time_delta_seconds")
        )
    )
)


# =========================================================
# 14. SPEED DIFFERENCE
# =========================================================

df = df.withColumn(
    "speed_difference_mps",
    F.when(
        F.col("trajectory_step_valid")
        &
        F.col("velocity_mps").isNotNull()
        &
        F.col("derived_speed_mps").isNotNull(),
        (
            F.col("derived_speed_mps")
            -
            F.col("velocity_mps")
        )
    )
)


# =========================================================
# 15. SPEED ERROR RATIO
# =========================================================

df = df.withColumn(
    "speed_error_ratio",
    F.when(
        F.col("trajectory_step_valid")
        &
        F.col("velocity_mps").isNotNull()
        &
        (F.col("velocity_mps") > 0),
        F.abs(
            F.col("derived_speed_mps")
            -
            F.col("velocity_mps")
        )
        /
        F.col("velocity_mps")
    )
)


df = df.withColumn(
    "speed_consistency_status",
    F.when(
        F.col("speed_error_ratio").isNull(),
        F.lit("UNKNOWN")
    )
    .when(
        F.col("speed_error_ratio")
        <= SPEED_ERROR_THRESHOLD,
        F.lit("CONSISTENT")
    )
    .otherwise(
        F.lit("REVIEW")
    )
)


# =========================================================
# 16. HORIZONTAL ACCELERATION
# =========================================================

df = df.withColumn(
    "acceleration_mps2",
    F.when(
        F.col("trajectory_step_valid")
        &
        F.col("previous_velocity_mps").isNotNull()
        &
        F.col("velocity_mps").isNotNull(),
        (
            F.col("velocity_mps")
            -
            F.col("previous_velocity_mps")
        )
        /
        F.col("time_delta_seconds")
    )
)


# =========================================================
# 17. ALTITUDE CHANGE
# =========================================================

df = df.withColumn(
    "altitude_delta_m",
    F.when(
        F.col("trajectory_step_valid")
        &
        F.col("previous_altitude_m").isNotNull()
        &
        F.col("baro_altitude_m").isNotNull(),
        (
            F.col("baro_altitude_m")
            -
            F.col("previous_altitude_m")
        )
    )
)


# =========================================================
# 18. VERTICAL ACCELERATION
# =========================================================

df = df.withColumn(
    "vertical_acceleration_mps2",
    F.when(
        F.col("trajectory_step_valid")
        &
        F.col("previous_vertical_rate_mps").isNotNull()
        &
        F.col("vertical_rate_mps").isNotNull(),
        (
            F.col("vertical_rate_mps")
            -
            F.col("previous_vertical_rate_mps")
        )
        /
        F.col("time_delta_seconds")
    )
)


# =========================================================
# 19. HEADING CHANGE
# =========================================================
#
# Circular angular difference.
#
# 359° → 1° = 2°
# =========================================================

heading_difference_expr = F.abs(
    F.pmod(
        (
            F.col("true_track")
            -
            F.col("previous_true_track")
            +
            F.lit(540.0)
        ),
        F.lit(360.0)
    )
    -
    F.lit(180.0)
)


df = df.withColumn(
    "heading_change_deg",
    F.when(
        F.col("trajectory_step_valid")
        &
        F.col("previous_true_track").isNotNull()
        &
        F.col("true_track").isNotNull(),
        heading_difference_expr
    )
)


# =========================================================
# 20. TURN RATE
# =========================================================

df = df.withColumn(
    "turn_rate_deg_per_sec",
    F.when(
        F.col("trajectory_step_valid")
        &
        F.col("heading_change_deg").isNotNull(),
        F.col("heading_change_deg")
        /
        F.col("time_delta_seconds")
    )
)


# =========================================================
# 21. H3 TRANSITION
# =========================================================

df = (
    df

    .withColumn(
        "h3_from_res_8",
        F.when(
            F.col("trajectory_step_valid"),
            F.col("previous_h3_res_8")
        )
    )

    .withColumn(
        "h3_to_res_8",
        F.when(
            F.col("trajectory_step_valid"),
            F.col("h3_res_8")
        )
    )

    .withColumn(
        "h3_transition",
        F.when(
            F.col("trajectory_step_valid")
            &
            F.col("previous_h3_res_8").isNotNull()
            &
            F.col("h3_res_8").isNotNull()
            &
            (
                F.col("previous_h3_res_8")
                !=
                F.col("h3_res_8")
            ),
            F.lit(True)
        )
        .otherwise(
            F.lit(False)
        )
    )
)


# =========================================================
# 22. H3 GRID DISTANCE
# =========================================================

df = df.withColumn(
    "h3_grid_distance",
    F.when(
        F.col("trajectory_step_valid")
        &
        F.col("previous_h3_res_8").isNotNull()
        &
        F.col("h3_res_8").isNotNull(),
        F.expr(
            "h3_distance("
            "previous_h3_res_8, "
            "h3_res_8"
            ")"
        )
    )
)


# =========================================================
# 23. FLIGHT SESSION
# =========================================================

df = df.withColumn(
    "telemetry_gap",
    F.when(
        F.col("time_delta_seconds")
        >
        SESSION_GAP_SECONDS,
        F.lit(True)
    )
    .otherwise(
        F.lit(False)
    )
)


df = df.withColumn(
    "new_session_flag",
    F.when(
        F.col("previous_event_time").isNull(),
        F.lit(1)
    )
    .when(
        F.col("time_delta_seconds")
        >
        SESSION_GAP_SECONDS,
        F.lit(1)
    )
    .otherwise(
        F.lit(0)
    )
)


# =========================================================
# 24. SESSION NUMBER
# =========================================================

session_number_window = (
    Window
    .partitionBy("icao24")
    .orderBy(
        F.col("event_time").asc(),
        F.col("observation_id").asc()
    )
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)


df = df.withColumn(
    "session_number",
    F.sum(
        "new_session_flag"
    ).over(
        session_number_window
    )
)


# =========================================================
# 25. SESSION ORDERED WINDOWS
# =========================================================

session_ordered_window = (
    Window
    .partitionBy(
        "icao24",
        "session_number"
    )
    .orderBy(
        F.col("event_time").asc(),
        F.col("observation_id").asc()
    )
    .rowsBetween(
        Window.unboundedPreceding,
        Window.unboundedFollowing
    )
)

session_reverse_window = (
    Window
    .partitionBy(
        "icao24",
        "session_number"
    )
    .orderBy(
        F.col("event_time").desc(),
        F.col("observation_id").desc()
    )
    .rowsBetween(
        Window.unboundedPreceding,
        Window.unboundedFollowing
    )
)


# =========================================================
# 26. SESSION METADATA
# =========================================================

df = (
    df

    .withColumn(
        "session_start",
        F.first(
            "event_time"
        ).over(
            session_ordered_window
        )
    )

    .withColumn(
        "session_end",
        F.first(
            "event_time"
        ).over(
            session_reverse_window
        )
    )

    .withColumn(
        "session_observation_count",
        F.count("*").over(
            session_ordered_window
        )
    )
)


# =========================================================
# 27. SESSION DURATION
# =========================================================

df = df.withColumn(
    "session_duration_seconds",
    (
        F.col("session_end").cast("long")
        -
        F.col("session_start").cast("long")
    ).cast("double")
)


df = df.withColumn(
    "session_duration_minutes",
    F.col("session_duration_seconds")
    /
    60.0
)

# =========================================================
# 28. CREATE TRAJECTORY SEGMENT
# =========================================================
#
# A segment starts when:
#
#   1. first observation
#   2. invalid time order
#   3. gap > MAX_KINEMATIC_GAP_SECONDS
#
# IMPORTANT:
# TOO_CLOSE does NOT create a segment break.
#
# This lets us preserve the aircraft trajectory even when
# an isolated observation arrives too quickly.
# =========================================================

df = df.withColumn(
    "new_trajectory_segment_flag",
    F.when(
        F.col("previous_event_time").isNull(),
        F.lit(1)
    )
    .when(
        ~F.col("trajectory_step_valid"),
        F.lit(1)
    )
    .otherwise(
        F.lit(0)
    )
)

# =========================================================
# 29. TRAJECTORY SEGMENT NUMBER
# =========================================================

trajectory_segment_window = (
    Window
    .partitionBy("icao24")
    .orderBy(
        F.col("event_time").asc(),
        F.col("observation_id").asc()
    )
    .rowsBetween(
        Window.unboundedPreceding,
        Window.currentRow
    )
)


df = df.withColumn(
    "trajectory_segment_number",
    F.sum(
        "new_trajectory_segment_flag"
    ).over(
        trajectory_segment_window
    )
)


# =========================================================
# 30. STABLE TRAJECTORY SEGMENT ID
# =========================================================

df = df.withColumn(
    "trajectory_segment_id",
    F.sha2(
        F.concat_ws(
            "||",
            F.col("icao24"),
            F.col(
                "trajectory_segment_number"
            ).cast("string")
        ),
        256
    )
)


# =========================================================
# 31. SEGMENT ORDERED WINDOWS
# =========================================================

segment_ordered_window = (
    Window
    .partitionBy(
        "icao24",
        "trajectory_segment_number"
    )
    .orderBy(
        F.col("event_time").asc(),
        F.col("observation_id").asc()
    )
    .rowsBetween(
        Window.unboundedPreceding,
        Window.unboundedFollowing
    )
)

segment_reverse_window = (
    Window
    .partitionBy(
        "icao24",
        "trajectory_segment_number"
    )
    .orderBy(
        F.col("event_time").desc(),
        F.col("observation_id").desc()
    )
    .rowsBetween(
        Window.unboundedPreceding,
        Window.unboundedFollowing
    )
)


# =========================================================
# 32. SEGMENT START / END
# =========================================================

df = (
    df

    .withColumn(
        "segment_start",
        F.first(
            "event_time"
        ).over(
            segment_ordered_window
        )
    )

    .withColumn(
        "segment_end",
        F.first(
            "event_time"
        ).over(
            segment_reverse_window
        )
    )

    .withColumn(
        "segment_start_latitude",
        F.first(
            "latitude",
            ignorenulls=True
        ).over(
            segment_ordered_window
        )
    )

    .withColumn(
        "segment_start_longitude",
        F.first(
            "longitude",
            ignorenulls=True
        ).over(
            segment_ordered_window
        )
    )

    .withColumn(
        "segment_end_latitude",
        F.first(
            "latitude",
            ignorenulls=True
        ).over(
            segment_reverse_window
        )
    )
    .withColumn(
        "segment_end_longitude",
        F.first(
            "longitude",
            ignorenulls=True
        ).over(
            segment_reverse_window
        )
    )

    .withColumn(
        "segment_observation_count",
        F.count("*").over(
            segment_ordered_window
        )
    )
)


# =========================================================
# 33. SEGMENT DURATION
# =========================================================

df = df.withColumn(
    "segment_duration_seconds",
    (
        F.col("segment_end").cast("long")
        -
        F.col("segment_start").cast("long")
    ).cast("double")
)


df = df.withColumn(
    "segment_duration_minutes",
    F.col("segment_duration_seconds")
    /
    60.0
)


# =========================================================
# 34. SEGMENT PATH LENGTH
# =========================================================
#
# Only valid trajectory steps contribute.
# =========================================================

df = df.withColumn(
    "segment_path_length_km",
    F.sum(
        F.when(
            F.col("trajectory_step_valid"),
            F.coalesce(
                F.col("step_distance_km"),
                F.lit(0.0)
            )
        ).otherwise(
            F.lit(0.0)
        )
    ).over(
        segment_ordered_window
    )
)


# =========================================================
# 35. SEGMENT NET DISPLACEMENT
# =========================================================

segment_lat1 = F.radians(
    F.col("segment_start_latitude")
)

segment_lon1 = F.radians(
    F.col("segment_start_longitude")
)

segment_lat2 = F.radians(
    F.col("segment_end_latitude")
)

segment_lon2 = F.radians(
    F.col("segment_end_longitude")
)

segment_dlat = (
    segment_lat2
    -
    segment_lat1
)

segment_dlon = (
    segment_lon2
    -
    segment_lon1
)

segment_a = (
    F.pow(
        F.sin(segment_dlat / 2),
        2
    )
    +
    F.cos(segment_lat1)
    * F.cos(segment_lat2)
    * F.pow(
        F.sin(segment_dlon / 2),
        2
    )
)

segment_a_safe = F.least(
    F.lit(1.0),
    F.greatest(
        F.lit(0.0),
        segment_a
    )
)

segment_net_displacement_expr = (
    F.lit(2.0)
    * F.lit(EARTH_RADIUS_KM)
    * F.asin(
        F.sqrt(segment_a_safe)
    )
)


df = df.withColumn(
    "segment_net_displacement_km",
    F.when(
        F.col("segment_start_latitude").isNotNull()
        &
        F.col("segment_start_longitude").isNotNull()
        &
        F.col("segment_end_latitude").isNotNull()
        &
        F.col("segment_end_longitude").isNotNull(),
        segment_net_displacement_expr
    )
)


# =========================================================
# 36. SEGMENT TRAJECTORY EFFICIENCY
# =========================================================
#
# Theoretically:
#
#   0 <= efficiency <= 1
#
# Do NOT silently clamp this value.
# Instead, we verify the invariant after writing.
# =========================================================

df = df.withColumn(
    "segment_trajectory_efficiency",
    F.when(
        F.col("segment_path_length_km") > 0,
        F.col("segment_net_displacement_km")
        /
        F.col("segment_path_length_km")
    )
)


# =========================================================
# 37. SEGMENT QUALITY
# =========================================================

df = df.withColumn(
    "segment_quality",
    F.when(
        F.col("segment_observation_count") < 2,
        F.lit("SINGLE_OBSERVATION")
    )
    .when(
        F.col("segment_path_length_km") <= 0,
        F.lit("NO_MOVEMENT")
    )
    .otherwise(
        F.lit("USABLE")
    )
)


# =========================================================
# 38. FLIGHT SESSION ID
# =========================================================

df = df.withColumn(
    "flight_session_id",
    F.sha2(
        F.concat_ws(
            "||",
            F.col("icao24"),
            F.col("session_start").cast("long")
        ),
        256
    )
)


# =========================================================
# 39. TRAJECTORY METADATA
# =========================================================

df = (
    df

    .withColumn(
        "trajectory_version",
        F.lit("trajectory-v3")
    )

    .withColumn(
        "trajectory_enriched_at",
        F.current_timestamp()
    )

    .withColumn(
        "event_date",
        F.to_date(
            F.col("event_time")
        )
    )
)


# =========================================================
# 40. DROP INTERNAL COLUMNS
# =========================================================

df_final = df.drop(
    "new_session_flag",
    "new_trajectory_segment_flag",
    "session_number",
    "trajectory_segment_number"
)


# =========================================================
# 41. WRITE TRAJECTORY DELTA
# =========================================================
#
# Rebuild deterministically from spatial-indexed Silver.
# =========================================================

(
    df_final
    .write
    .format("delta")
    .mode("overwrite")
    .option(
        "overwriteSchema",
        "true"
    )
    .save(
        TRAJECTORY_PATH
    )
)

# =========================================================
# 42. RELOAD OUTPUT
# =========================================================

result = (
    spark.read
    .format("delta")
    .load(
        TRAJECTORY_PATH
    )
)


# =========================================================
# 43. BASIC DATASET METRICS
# =========================================================

total_rows = result.count()

distinct_aircraft = (
    result
    .select("icao24")
    .distinct()
    .count()
)

distinct_sessions = (
    result
    .select("flight_session_id")
    .distinct()
    .count()
)

distinct_segments = (
    result
    .select("trajectory_segment_id")
    .distinct()
    .count()
)

print(
    f"Trajectory rows       : {total_rows}"
)

print(
    f"Distinct aircraft     : {distinct_aircraft}"
)

print(
    f"Distinct sessions     : {distinct_sessions}"
)

print(
    f"Distinct segments     : {distinct_segments}"
)


# =========================================================
# 44. TRAJECTORY STEP STATUS
# =========================================================

display(
    result
    .groupBy(
        "trajectory_step_status"
    )
    .count()
    .orderBy(
        "trajectory_step_status"
    )
)


# =========================================================
# 45. STEP VALIDITY
# =========================================================

display(
    result
    .groupBy(
        "trajectory_step_valid"
    )
    .count()
)


# =========================================================
# 46. SPEED CONSISTENCY
# =========================================================

display(
    result
    .groupBy(
        "speed_consistency_status"
    )
    .count()
    .orderBy(
        "speed_consistency_status"
    )
)


# =========================================================
# 47. SEGMENT QUALITY
# =========================================================

display(
    result
    .groupBy(
        "segment_quality"
    )
    .count()
    .orderBy(
        "segment_quality"
    )
)


# =========================================================
# 48. SEGMENT EFFICIENCY VALIDATION
# =========================================================
#
# This is the invariant that previously exposed the bug.
# =========================================================

invalid_efficiency = (
    result
    .filter(
        F.col(
            "segment_trajectory_efficiency"
        ).isNotNull()
        &
        (
            (
                F.col(
                    "segment_trajectory_efficiency"
                ) < 0
            )
            |
            (
                F.col(
                    "segment_trajectory_efficiency"
                ) > 1.000001
            )
        )
    )
    .count()
)

print(
    "Invalid segment efficiency values: "
    f"{invalid_efficiency}"
)

if invalid_efficiency > 0:
    raise ValueError(
        "Trajectory invariant failed: "
        "segment_trajectory_efficiency "
        "must be between 0 and 1."
    )


# =========================================================
# 49. SEGMENT EFFICIENCY SUMMARY
# =========================================================

display(
    result
    .filter(
        F.col(
            "segment_trajectory_efficiency"
        ).isNotNull()
    )
    .select(
        "segment_trajectory_efficiency"
    )
    .summary()
)

invalid_efficiency = (
    result
    .filter(
        F.col("segment_trajectory_efficiency").isNotNull()
        &
        (
            (F.col("segment_trajectory_efficiency") < -1e-9)
            |
            (F.col("segment_trajectory_efficiency") > 1.000001)
        )
    )
    .count()
)

print(
    f"Invalid segment efficiency values: {invalid_efficiency}"
)

if invalid_efficiency > 0:
    raise ValueError(
        "Trajectory invariant failed: "
        "segment_trajectory_efficiency must be between 0 and 1."
    )
# =========================================================
# 50. TRAJECTORY SAMPLE
# =========================================================

display(
    result
    .select(
        "icao24",
        "event_time",

        "latitude",
        "longitude",

        "baro_altitude_m",
        "velocity_mps",
        "true_track",
        "vertical_rate_mps",

        "h3_res_8",

        "previous_event_time",
        "previous_latitude",
        "previous_longitude",

        "time_delta_seconds",

        "trajectory_step_status",
        "trajectory_step_valid",

        "step_distance_km",
        "derived_speed_mps",

        "speed_difference_mps",
        "speed_error_ratio",
        "speed_consistency_status",

        "acceleration_mps2",
        "altitude_delta_m",
        "vertical_acceleration_mps2",

        "heading_change_deg",
        "turn_rate_deg_per_sec",

        "h3_from_res_8",
        "h3_to_res_8",
        "h3_transition",
        "h3_grid_distance",

        "telemetry_gap",

        "flight_session_id",
        "session_start",
        "session_end",

        "trajectory_segment_id",
        "segment_start",
        "segment_end",
        "segment_duration_minutes",
        "segment_path_length_km",
        "segment_net_displacement_km",
        "segment_trajectory_efficiency",
        "segment_quality"
    )
    .orderBy(
        "icao24",
        "event_time"
    )
    .limit(100)
)

Trajectory rows       : 108103
Distinct aircraft     : 559
Distinct sessions     : 1898
Distinct segments     : 24480


trajectory_step_status,count
FIRST_OBSERVATION,559
GAP_TOO_LARGE,2392
INVALID_TIME_ORDER,1945
TOO_CLOSE,19584
VALID,83623


trajectory_step_valid,count
false,24480
true,83623


speed_consistency_status,count
CONSISTENT,80880
REVIEW,1543
UNKNOWN,25680


segment_quality,count
NO_MOVEMENT,461
SINGLE_OBSERVATION,9393
USABLE,98249


Invalid segment efficiency values: 0


summary,segment_trajectory_efficiency
count,98249
mean,0.9548080581381663
stddev,0.13298476873717294
min,0.0
25%,0.9903906980447886
50%,0.9999772195910325
75%,0.9999988057432203
max,1.0


Invalid segment efficiency values: 0


icao24,event_time,latitude,longitude,baro_altitude_m,velocity_mps,true_track,vertical_rate_mps,h3_res_8,previous_event_time,previous_latitude,previous_longitude,time_delta_seconds,trajectory_step_status,trajectory_step_valid,step_distance_km,derived_speed_mps,speed_difference_mps,speed_error_ratio,speed_consistency_status,acceleration_mps2,altitude_delta_m,vertical_acceleration_mps2,heading_change_deg,turn_rate_deg_per_sec,h3_from_res_8,h3_to_res_8,h3_transition,h3_grid_distance,telemetry_gap,flight_session_id,session_start,session_end,trajectory_segment_id,segment_start,segment_end,segment_duration_minutes,segment_path_length_km,segment_net_displacement_km,segment_trajectory_efficiency,segment_quality
00b22b,2026-09-22T08:00:54.000Z,-26.0281,29.6534,7696.2,237.92,87.4,5.85,615811529832398847,null,null,null,null,FIRST_OBSERVATION,false,null,null,null,null,UNKNOWN,null,null,null,null,null,null,null,false,null,false,0b2128ac94c39b61ccd16080e71b2c45ad41283601485aa27c66b83365cab144,2026-09-22T08:00:54.000Z,2026-09-22T08:06:39.000Z,d4b74733a497fb24007d42a9fb971273f7535ef37db94e344f408f680bea9445,2026-09-22T08:00:54.000Z,2026-09-22T08:06:39.000Z,5.75,84.91116593526196,84.91114112494327,0.9999997078085265,USABLE
00b22b,2026-09-22T08:01:10.000Z,-26.0266,29.6918,7795.26,238.43,87.4,5.53,615811529652043775,2026-09-22T08:00:54.000Z,-26.0281,29.6534,16.0,VALID,true,3.8404770497700302,240.02981561062688,1.5998156106268766,0.006709791597646591,CONSISTENT,0.03187500000000121,99.0600000000004,-0.019999999999999962,0.0,0.0,615811529832398847,615811529652043775,true,4,false,0b2128ac94c39b61ccd16080e71b2c45ad41283601485aa27c66b83365cab144,2026-09-22T08:00:54.000Z,2026-09-22T08:06:39.000Z,d4b74733a497fb24007d42a9fb971273f7535ef37db94e344f408f680bea9445,2026-09-22T08:00:54.000Z,2026-09-22T08:06:39.000Z,5.75,84.91116593526196,84.91114112494327,0.9999997078085265,USABLE
00b22b,2026-09-22T08:01:57.000Z,-26.0219,29.8039,8031.48,242.05,87.32,5.2,615811528716713983,2026-09-22T08:01:10.000Z,-26.0266,29.6918,47.0,VALID,true,11.213295291984563,238.58075089328858,-3.4692491067114304,0.014332778792445487,CONSISTENT,0.07702127659574477,236.21999999999935,-0.007021276595744683,0.08000000000004093,0.001702127659575339,615811529652043775,615811528716713983,true,13,false,0b2128ac94c39b61ccd16080e71b2c45ad41283601485aa27c66b83365cab144,2026-09-22T08:00:54.000Z,2026-09-22T08:06:39.000Z,d4b74733a497fb24007d42a9fb971273f7535ef37db94e344f408f680bea9445,2026-09-22T08:00:54.000Z,2026-09-22T08:06:39.000Z,5.75,84.91116593526196,84.91114112494327,0.9999997078085265,USABLE
00b22b,2026-09-22T08:02:29.000Z,-26.0187,29.8813,8199.12,243.57,87.46,4.88,615811528607662079,2026-09-22T08:01:57.000Z,-26.0219,29.8039,32.0,VALID,true,7.74230310715947,241.94697209873345,-1.6230279012665392,0.0066634967412511365,CONSISTENT,0.04749999999999943,167.64000000000124,-0.010000000000000009,0.13999999999998636,0.004374999999999574,615811528716713983,615811528607662079,true,9,false,0b2128ac94c39b61ccd16080e71b2c45ad41283601485aa27c66b83365cab144,2026-09-22T08:00:54.000Z,2026-09-22T08:06:39.000Z,d4b74733a497fb24007d42a9fb971273f7535ef37db94e344f408f680bea9445,2026-09-22T08:00:54.000Z,2026-09-22T08:06:39.000Z,5.75,84.91116593526196,84.91114112494327,0.9999997078085265,USABLE
00b22b,2026-09-22T08:03:06.000Z,-26.0148,29.9719,8366.76,246.19,87.25,4.88,615811545001099263,2026-09-22T08:02:29.000Z,-26.0187,29.8813,37.0,VALID,true,9.063773868067317,244.9668612991167,-1.223138700883311,0.004968271257497506,CONSISTENT,0.07081081081081093,167.63999999999942,0.0,0.21000000000003638,0.0056756756756766585,615811528607662079,615811545001099263,true,11,false,0b2128ac94c39b61ccd16080e71b2c45ad41283601485aa27c66b83365cab144,2026-09-22T08:00:54.000Z,2026-09-22T08:06:39.000Z,d4b74733a497fb24007d42a9fb971273f7535ef37db94e344f408f680bea9445,2026-09-22T08:00:54.000Z,2026-09-22T08:06:39.000Z,5.75,84.91116593526196,84.91114112494327,0.9999997078085265,USABLE
00b22b,2026-09-22T08:03:35.000Z,-26.0117,30.0439,8534.4,246.7,87